# Speech Emotion Recognition using MFCC and LSTM

This notebook reproduces the training workflow from the original Group 15 case study. It expects the TESS dataset to be available locally. Set `TESS_DATA_DIR` to the dataset directory or place the data under `data/`.

**Label check:** The original notebook extracts seven labels from filenames: `angry`, `disgust`, `fear`, `happy`, `neutral`, `ps`, and `sad`. The presentation describes `ps` as “Surprised”; verify the dataset naming before relabeling it.


In [ ]:
#install
%pip install librosa

In [ ]:
pip install tensorflow

In [ ]:
#modules
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
import librosa 
import librosa.display
from IPython.display import Audio
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Scan dataset directory, collect all .wav file paths, and extract emotion labels from filenames
data_dir = os.environ.get('TESS_DATA_DIR', 'data/TESS Toronto emotional speech set data')
print("Path exists:", os.path.exists(data_dir))

paths = []
labels = []

for root, dirs, files in os.walk(data_dir):
    for file in files:
        if file.endswith('.wav'):
            paths.append(os.path.join(root, file))
            label = file.split('_')[-1].split('.')[0].lower()
            labels.append(label)

In [ ]:
# Convert paths and labels into a Pandas DataFrame
speech = pd.DataFrame({'speech': paths,
                      'label' : labels})
print(f"Total files: {len(speech)}")
speech.head()

In [ ]:
speech['label'].value_counts()

In [ ]:
sns.countplot(data=speech, x='label')

In [ ]:
# Extract MFCC audio features from each file and reshape them into a 3D array for deep learning models
def extract_mfcc(filename):
    y, sr = librosa.load(filename, duration=3, offset=0.5)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    return mfcc

X_mfcc = speech['speech'].apply(lambda x: extract_mfcc(x))
X = np.array([x for x in X_mfcc])
X = np.expand_dims(X,-1)

print("Input shape ready for model:", X.shape)

In [ ]:
# 1. Pick a sample file from your DataFrame
sample_path = speech['speech'].iloc[0]
sample_label = speech['label'].iloc[0]

# 2. Load the audio clip
y, sr = librosa.load(sample_path, duration=3, offset=0.5)

# 3. Compute MFCCs (keep time steps intact for visualization)
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)

# 4. Plot the spectrogram
plt.figure(figsize=(10, 4))
librosa.display.specshow(mfcc, sr=sr, x_axis='time')

plt.colorbar(format='%+2.0f dB')
plt.title(f"MFCC Spectrogram - Emotion: {sample_label.capitalize()}")
plt.xlabel("Time (seconds)")
plt.ylabel("MFCC Coefficients (0-40)")
plt.tight_layout()
plt.show()

In [ ]:
# Convert text emotion labels into a one-hot encoded binary matrix (0s and 1s) for neural network targets
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()
y = encoder.fit_transform(speech[['label']]).toarray()

print("Encoded Target Shape:", y.shape)  # Expected: (2800, 7)
print("Detected Emotion Classes:", encoder.categories_[0])

In [ ]:
# Split features and labels into 80% training and 20% testing sets while keeping emotion class proportions balanced
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42, stratify = y
)

print("Training features shape:", X_train.shape)  # Expected: (2240, 40, 1)
print("Testing features shape: ", X_test.shape)  # Expected: (560, 40, 1)

In [ ]:
# 1. Build an LSTM neural network to analyze time-sequence patterns in the 40 MFCC audio features
# 2. Add Dense layers with Dropout (20%) to refine learned patterns and prevent overfitting
# 3. Compile the model with Adam optimizer and Softmax activation for multi-class emotion classification
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout

model = Sequential([
    LSTM(128, input_shape=(40, 1), return_sequences=False),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(y.shape[1], activation='softmax'),
])

model.compile(
    loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy']
)
model.summary()

In [ ]:
# 1. Train the model for 50 full passes (epochs) over the training dataset
# 2. Process data in mini-batches of 64 samples and shuffle order each epoch
# 3. Evaluate loss and accuracy on the validation set (X_test, y_test) after every pass
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=50,
    batch_size=64,
    shuffle=True,
)

In [ ]:
# Create side-by-side plots comparing training and validation accuracy and loss across all epochs
plt.figure(figsize=(12, 4))

# Plot Accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Curve')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.show()

In [ ]:
# Evaluate model performance on unseen test data using a detailed classification report and a visual confusion matrix heatmap
from sklearn.metrics import classification_report, confusion_matrix

# Predict on unseen test data only
y_pred_probs = model.predict(X_test)

# Convert one-hot vectors to index labels
y_true_test = np.argmax(y_test, axis=1)
y_pred_test = np.argmax(y_pred_probs, axis=1)

class_names = encoder.categories_[0]

# Print evaluation report
print("=== Unseen Test Set Classification Report ===")
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

# Plot Confusion Matrix
cm = confusion_matrix(y_true_test, y_pred_test)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names,
)
plt.title('Test Set Confusion Matrix (560 Samples)')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# Define a function to play an audio file inline, predict its emotion using the trained model, and plot a confidence bar chart
from IPython.display import Audio, display

def play_and_predict(audio_path, model, encoder):
    print("=" * 45)
    print("🔊 AUDIO PLAYER:")
    # 1. Display inline audio player widget
    display(Audio(filename=audio_path))
    
    # 2. Extract MFCC features
    y, sr = librosa.load(audio_path, duration=3, offset=0.5)
    mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40).T, axis=0)
    features = np.expand_dims(mfcc, axis=(0, -1))
    
    # 3. Predict emotion probabilities
    predictions = model.predict(features, verbose=0)[0]
    class_idx = np.argmax(predictions)
    predicted_emotion = encoder.categories_[0][class_idx]
    confidence = predictions[class_idx] * 100
    
    # 4. Print core prediction result
    print(f"🎯 PREDICTED EMOTION: {predicted_emotion.upper()}")
    print(f"📊 CONFIDENCE SCORE:  {confidence:.2f}%")
    print("=" * 45)
    
    # 5. Optional: Display probability bar chart across all emotions
    plt.figure(figsize=(8, 3))
    emotions = encoder.categories_[0]
    plt.bar(emotions, predictions * 100, color='skyblue')
    plt.ylabel('Confidence (%)')
    plt.title('Emotion Probability Distribution')
    plt.xticks(rotation=30)
    plt.ylim(0, 100)
    plt.tight_layout()
    plt.show()

In [ ]:
random_sample = speech.sample(1).iloc[0]

print(f"Actual Ground Truth Label: {random_sample['label'].upper()}")
play_and_predict(random_sample['speech'], model, encoder)

In [ ]:
# Loop through each unique emotion class, randomly select a sample audio file, and test the model's prediction
for emotion_name in speech['label'].unique():
    sample = speech[speech['label'] == emotion_name].sample(50).iloc[0]
    print(f"\nTarget Emotion: {emotion_name.upper()}")
    play_and_predict(sample['speech'], model, encoder)

In [ ]:
# Evaluate final training and testing accuracy numbers and run an overfitting check
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Training Accuracy: {train_acc * 100:.2f}%")
print(f"Testing Accuracy:  {test_acc * 100:.2f}%")

In [ ]:
# Calculate performance gap
acc_gap = (train_acc - test_acc) * 100
print(f"Accuracy Gap:      {acc_gap:.2f}%")


## Save the trained model

Save the model so the Streamlit app can load it without retraining. Run this after model training.


In [ ]:
import os
os.makedirs("models", exist_ok=True)
model.save("models/speech_emotion_lstm.keras")
print("Model saved to models/speech_emotion_lstm.keras")
